# Webcam Classifier Training

Train a binary classifier to detect **ambulance** vs **regular vehicles** from webcam images.

This notebook:
1. Loads image data from `Dataset/emergency-vehicles/`
2. Extracts features (color histograms, edges, etc.)
3. Trains a classifier (RandomForest or SVM)
4. Saves the model to `Trained_Model/webcam_classifier.pkl`

In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded successfully")

## Step 1: Load Dataset

In [ ]:
# Path to emergency vehicles dataset
train_data_path = Path('../Dataset/emergency-vehicles/train')
train_csv_path = Path('../Dataset/emergency-vehicles/train.csv')

# Load image labels
df = pd.read_csv(train_csv_path)
print(f"Dataset shape: {df.shape}")
print(df.head(10))
print(f"\nClass distribution:\n{df['emergency_or_not'].value_counts()}")

## Step 2: Feature Extraction

Extract visual features from images:
- Color histogram (RGB)
- Edge detection (Canny)
- Texture features

In [ ]:
def extract_features(image_path, img_size=(128, 128)):
    """Extract features from image."""
    try:
        img = cv2.imread(str(image_path))
        if img is None:
            return None
        
        # Resize
        img = cv2.resize(img, img_size)
        
        # Color histogram
        hist_b = cv2.calcHist([img], [0], None, [32], [0, 256])
        hist_g = cv2.calcHist([img], [1], None, [32], [0, 256])
        hist_r = cv2.calcHist([img], [2], None, [32], [0, 256])
        hist = np.concatenate([hist_b.flatten(), hist_g.flatten(), hist_r.flatten()])
        
        # Grayscale for edge detection
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 50, 150)
        edge_feature = np.array([edges.mean(), edges.std(), edges.max()])
        
        # Combine features
        features = np.concatenate([hist, edge_feature])
        return features
    except Exception as e:
        print(f"Error extracting features from {image_path}: {e}")
        return None

print("Feature extraction function defined")

In [ ]:
# Extract features for all images
X = []
y = []

for idx, row in df.iterrows():
    img_path = train_data_path / row['image_names']
    features = extract_features(img_path)
    
    if features is not None:
        X.append(features)
        y.append(row['emergency_or_not'])
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1} images")

X = np.array(X)
y = np.array(y)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Label distribution: {pd.Series(y).value_counts().to_dict()}")

## Step 3: Train-Test Split

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTraining labels: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Test labels: {pd.Series(y_test).value_counts().to_dict()}")

## Step 4: Model Training

In [ ]:
# Train RandomForest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=15, n_jobs=-1)
print("Training classifier...")
clf.fit(X_train, y_train)
print("Training complete!")

# Predictions
y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)

# Evaluate
acc = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {acc:.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=['Regular Vehicle', 'Emergency Vehicle'])}")

## Step 5: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Regular', 'Emergency'],
            yticklabels=['Regular', 'Emergency'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print(f"True Negatives: {cm[0, 0]}")
print(f"False Positives: {cm[0, 1]}")
print(f"False Negatives: {cm[1, 0]}")
print(f"True Positives: {cm[1, 1]}")

## Step 6: Save Model

In [ ]:
# Save model
model_save_path = Path('../Trained_Model/webcam_classifier.pkl')
model_save_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(clf, model_save_path)
print(f"Model saved to {model_save_path}")

# Save feature extractor metadata
metadata = {
    'accuracy': float(acc),
    'n_features': X.shape[1],
    'img_size': (128, 128),
    'classes': ['Regular Vehicle', 'Emergency Vehicle'],
    'training_samples': len(X_train),
    'test_samples': len(X_test)
}

import json
metadata_path = Path('../Trained_Model/webcam_classifier_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to {metadata_path}")
print(f"\nModel Info: {metadata}")

## Step 7: Test on New Image

In [ ]:
def predict_vehicle_type(image_path, model_path='../Trained_Model/webcam_classifier.pkl'):
    """Predict vehicle type from image."""
    # Load model
    model = joblib.load(model_path)
    
    # Extract features
    features = extract_features(image_path)
    if features is None:
        return None, None
    
    # Predict
    pred = model.predict([features])[0]
    proba = model.predict_proba([features])[0]
    
    class_names = ['Regular Vehicle', 'Emergency Vehicle']
    return class_names[pred], proba[pred]

# Test on first test image
test_img = train_data_path / df.iloc[0]['image_names']
if test_img.exists():
    pred_class, pred_conf = predict_vehicle_type(str(test_img))
    print(f"Test image: {df.iloc[0]['image_names']}")
    print(f"Predicted: {pred_class}")
    print(f"Confidence: {pred_conf:.4f}")
else:
    print(f"Test image not found: {test_img}")

## Summary

Model berhasil dilatih dan disimpan di `Trained_Model/webcam_classifier.pkl`.

Untuk menggunakan model ini di `app.py` (Webcam AI view):
```python
import joblib
model = joblib.load('Trained_Model/webcam_classifier.pkl')
features = extract_features(image)
pred_class, conf = model.predict([features])[0], model.predict_proba([features])[0].max()
```